- Sets up the volume and subfolder to store the raw files and extracted images.
- Required permissions: 
  - **Catalog**: `USE CATALOG`
  - **Schema**: `USE SCHEMA`, `CREATE VOLUME`
  - **Volume**: `WRITE VOLUME` (for creating directories and uploading files)
- Note: You can have `WRITE VOLUME` at the schema or catalog level, and it will automatically apply to all current and future volumes through privilege inheritance.

In [0]:
import shutil
import os


# check if the volume exists:
def volume_exists(catalog, schema, volume):
    """Check if volume exists by attempting to describe it."""
    try:
        spark.sql(f"DESCRIBE VOLUME {catalog}.{schema}.{volume}").collect()
        return True
    except Exception:
        return False


def create_volume(catalog, schema, volume):
    """Create volume if it doesn't exist."""
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
    print(f"✅ Volume {catalog}.{schema}.{volume} created.")
    return True


def create_directory(catalog, schema, volume, directory):
    """Create directory under volume if it doesn't exist."""
    path = f"/Volumes/{catalog}/{schema}/{volume}/{directory}"
    try:
        os.makedirs(path, exist_ok=True)
        print(f"✅ Directory created: {path}")
        return True
    except Exception as e:
        print(f"❌ Failed to create directory {path}: {e}")
        return False

def file_exists(catalog, schema, volume, folder, file):
    """Check if file exists under volume."""
    path = f"/Volumes/{catalog}/{schema}/{volume}/{folder}/{file}"
    return os.path.exists(path)


def upload_file_to_volume(source_file, catalog, schema, volume, dest_folder, dest_filename):
    """
    Upload a file from local path to Unity Catalog volume.
    
    Args:
        source_file: Source file path (e.g., "/data/myfile.pdf")
        catalog, schema, volume: UC volume location
        dest_folder: Subfolder within volume (e.g., "input/subfolder")
        dest_filename: Name for the file in volume
    """
    # Ensure target directory exists
    dest_dir = f"/Volumes/{catalog}/{schema}/{volume}/{dest_folder}"

    
    # Full destination path
    destination = f"{dest_dir}/{dest_filename}"
    
    try:
        shutil.copy2(source_file, destination)
        print(f"✅ File uploaded: {source_file} → {destination}")
        return True
    except Exception as e:
        print(f"❌ Failed to upload file: {e}")
        return False


In [0]:
source_file = "./data/large-language-model-starter-kit.pdf" # local file


catalog = "workspace"
schema = "bronze"

# destination for the uploaded file
volume = "rag"
input_folder = "input" 
input_file = "large-language-model-starter-kit.pdf" 
output_folder = "output" # for OCR images extracted from the file

# input
if file_exists(catalog, schema, volume, input_folder, input_file):
    print(f"File {catalog}.{schema}.{volume}/{input_folder}/{input_file} already exists.")
else:
    if not volume_exists(catalog, schema, volume):
        create_volume(catalog, schema, volume)
    create_directory(catalog, schema, volume, input_folder)
    upload_file_to_volume(source_file, catalog, schema, volume, input_folder, input_file)
    print(f"File {catalog}.{schema}.{volume}/{input_folder}/{input_file} uploaded.")

# output
create_directory(catalog, schema, volume, output_folder)